In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
appointments_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.appointments")

patients_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.patients")

doctors_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.doctors")

In [0]:
display(appointments_df)

appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,ingestion_timestamp,source_file,load_date
A001,P034,D009,2023-08-09,2026-07-02T15:15:00.000Z,Therapy,Scheduled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A002,P032,D004,2023-06-09,2026-07-02T14:30:00.000Z,Therapy,No-show,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A003,P048,D004,2023-06-28,2026-07-02T08:00:00.000Z,Consultation,Cancelled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A004,P025,D006,2023-09-01,2026-07-02T09:15:00.000Z,Consultation,Cancelled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A005,P040,D003,2023-07-06,2026-07-02T12:45:00.000Z,Emergency,No-show,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A006,P045,D006,2023-06-19,2026-07-02T16:15:00.000Z,Checkup,Scheduled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A007,P001,D007,2023-04-09,2026-07-02T10:30:00.000Z,Consultation,Scheduled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A008,P016,D010,2023-05-24,2026-07-02T08:45:00.000Z,Consultation,Cancelled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A009,P039,D010,2023-03-05,2026-07-02T13:45:00.000Z,Follow-up,Scheduled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02
A010,P005,D003,2023-01-13,2026-07-02T15:30:00.000Z,Therapy,Completed,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02


In [0]:
appointments_df = appointments_df.dropDuplicates(["appointment_id"])

In [0]:
appointments_df = (

    appointments_df

    .withColumn("reason_for_visit",
                initcap(trim(col("reason_for_visit"))))

    .withColumn("status",
                initcap(trim(col("status"))))
)

In [0]:
appointments_df = appointments_df.join(

    patients_df.select("patient_id"),

    on="patient_id",

    how="inner"

)

In [0]:
appointments_df = appointments_df.join(

    doctors_df.select("doctor_id"),

    on="doctor_id",

    how="inner"

)

In [0]:
appointments_df = (

    appointments_df

    .withColumn(
        "completed_flag",
        when(col("status") == "Completed", 1).otherwise(0)
    )

    .withColumn(
        "cancelled_flag",
        when(col("status") == "Cancelled", 1).otherwise(0)
    )

    .withColumn(
        "no_show_flag",
        when(col("status") == "No-show", 1).otherwise(0)
    )

    .withColumn(
        "scheduled_flag",
        when(col("status") == "Scheduled", 1).otherwise(0)
    )

)

In [0]:
appointments_df = (

    appointments_df

    .withColumn(
        "appointment_year",
        year(col("appointment_date"))
    )

    .withColumn(
        "appointment_month",
        month(col("appointment_date"))
    )

    .withColumn(
        "appointment_day",
        dayofmonth(col("appointment_date"))
    )

    .withColumn(
        "appointment_weekday",
        date_format(col("appointment_date"), "EEEE")
    )
)

In [0]:
appointments_df = appointments_df.withColumn(

    "silver_load_timestamp",

    current_timestamp()

)

In [0]:
appointments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.appointments")

In [0]:

display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.appointments")
)

doctor_id,patient_id,appointment_id,appointment_date,appointment_time,reason_for_visit,status,ingestion_timestamp,source_file,load_date,completed_flag,cancelled_flag,no_show_flag,scheduled_flag,appointment_year,appointment_month,appointment_day,appointment_weekday,silver_load_timestamp
D003,P029,A012,2023-05-07,2026-07-02T10:00:00.000Z,Follow-up,Completed,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,1,0,0,0,2023,5,7,Sunday,2026-07-02T07:19:20.619Z
D006,P031,A044,2023-09-20,2026-07-02T12:30:00.000Z,Follow-up,Completed,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,1,0,0,0,2023,9,20,Wednesday,2026-07-02T07:19:20.619Z
D007,P032,A047,2023-05-02,2026-07-02T11:00:00.000Z,Therapy,Completed,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,1,0,0,0,2023,5,2,Tuesday,2026-07-02T07:19:20.619Z
D008,P045,A050,2023-08-16,2026-07-02T15:00:00.000Z,Consultation,No-show,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,0,0,1,0,2023,8,16,Wednesday,2026-07-02T07:19:20.619Z
D008,P013,A078,2023-09-17,2026-07-02T11:15:00.000Z,Consultation,No-show,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,0,0,1,0,2023,9,17,Sunday,2026-07-02T07:19:20.619Z
D007,P011,A099,2023-07-04,2026-07-02T15:00:00.000Z,Checkup,Completed,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,1,0,0,0,2023,7,4,Tuesday,2026-07-02T07:19:20.619Z
D008,P013,A124,2023-03-16,2026-07-02T17:15:00.000Z,Emergency,Cancelled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,0,1,0,0,2023,3,16,Thursday,2026-07-02T07:19:20.619Z
D005,P012,A140,2023-02-05,2026-07-02T15:15:00.000Z,Checkup,No-show,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,0,0,1,0,2023,2,5,Sunday,2026-07-02T07:19:20.619Z
D007,P012,A143,2023-09-21,2026-07-02T12:15:00.000Z,Checkup,Cancelled,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,0,1,0,0,2023,9,21,Thursday,2026-07-02T07:19:20.619Z
D003,P016,A159,2023-04-08,2026-07-02T16:15:00.000Z,Emergency,No-show,2026-07-02T06:36:45.904Z,appointments.csv,2026-07-02,0,0,1,0,2023,4,8,Saturday,2026-07-02T07:19:20.619Z


In [0]:
spark.sql("""
SELECT status, COUNT(*)
FROM workspace.silver.appointments
GROUP BY status
""").show()

+---------+--------+
|   status|COUNT(*)|
+---------+--------+
|Completed|      46|
|  No-show|      52|
|Cancelled|      51|
|Scheduled|      51|
+---------+--------+

